# AIQ — Test Notebook

Interactive testing of the AIQ pipeline. Run cells top-to-bottom.

## Cell 1 — Install & Import

In [ ]:
from aiq import analyze, AIQConfig, Pipeline
from aiq.loader import load_file, load_directory
from aiq.core.types import ChunkTag

print("AIQ imported successfully")

## Cell 2 — Simple Analysis (one-liner)

In [ ]:
result = analyze("All refunds are processed within 5 business days. Contact the billing team for help.")

print(f"Chunks:   {len(result.chunks)}")
print(f"Detected: {result.total_detected}")
print(f"Resolved: {result.total_resolved}")
print(f"Domain:   {result.domain_context.domain_type}")
print(f"Time:     {result.elapsed_seconds:.2f}s")

## Cell 3 — HTML Document with Issues

In [ ]:
html = """
<h1>Support Knowledge Base</h1>

<h2>Refund Policy</h2>
<p>All refunds are processed within 5 business days by the billing team.
For Enterprise customers, refunds over $500 require manager approval.</p>

<p>For questions, contact Sarah Johnson at sarah.johnson@acme.com or (555) 867-5309.</p>

<p><strong>INTERNAL NOTE:</strong> The actual SLA is 48 hours but we tell customers
5 days as a buffer. Do not share this with customers.</p>

<p>TODO: Add information about the new refund portal launching Q2 2026.</p>

<h2>Payment Methods</h2>
<table>
<tr><th>Method</th><th>Processing Time</th><th>Fee</th></tr>
<tr><td>Credit Card</td><td>Instant</td><td>2.9%</td></tr>
<tr><td>Bank Transfer</td><td>3-5 business days</td><td>Free</td></tr>
<tr><td>PayPal</td><td>Instant</td><td>3.5%</td></tr>
</table>

<h2>Troubleshooting</h2>
<p>The system seamlessly handles most edge cases automatically.
If the issue persists, escalate to the appropriate team.</p>

<p>Tracked in JIRA-1234. See #eng-backend for updates.</p>

<p><!-- Draft v2 - pending legal review --></p>
"""

result = analyze(html)

print(f"Chunks:   {len(result.chunks)}")
print(f"Detected: {result.total_detected}")
print(f"Domain:   {result.domain_context.domain_type}")
print(f"Acronyms: {result.domain_context.acronyms}")
print()

# Show findings
a31 = result.module_outputs.get("A31")
if a31 and a31.findings:
    print(f"Governance findings ({len(a31.findings)}):")
    for f in a31.findings:
        print(f"  [{f.tag.value}] {f.reason}")

print()
print("Chunk tags:")
for c in result.chunks:
    tag = c.tag.value
    if tag != "content":
        print(f"  [{tag.upper()}] {c.tag_reason}")
    else:
        print(f"  [SAFE] {c.heading or c.content[:60]}")

## Cell 4 — Multi-Document with Metadata

In [ ]:
docs = [
    {
        "id": "refund_policy",
        "title": "Refund Policy",
        "text": "<h1>Refund Policy</h1><p>All refunds are processed within 5 business days by the billing team.</p>",
        "metadata": {
            "author": "Product Team",
            "last_modified": "2026-04-15",
            "status": "published",
        },
    },
    {
        "id": "billing_faq",
        "title": "Billing FAQ",
        "text": "<h1>Billing FAQ</h1><p>Refunds typically take 10 business days. INTERNAL NOTE: do not share actual SLA.</p>",
        "metadata": {
            "author": "Intern",
            "last_modified": "2024-06-01",
            "status": "draft",
        },
    },
    {
        "id": "api_docs",
        "title": "API Documentation",
        "text": "<h1>API</h1><p>The REST API uses OAuth 2.0. Rate limits: 100 req/min (Starter), 1000 req/min (Enterprise).</p>",
        "metadata": {
            "author": "Engineering",
            "last_modified": "2026-04-20",
            "status": "published",
        },
    },
]

config = AIQConfig(
    pii_mode="smart",
    freshness_threshold_days=365,  # flag content older than 1 year
)

result = analyze(docs, config=config)

print(f"Documents:  {len(result.documents)}")
print(f"Chunks:     {len(result.chunks)}")
print(f"Detected:   {result.total_detected}")
print(f"Domain:     {result.domain_context.domain_type}")
print()

for c in result.chunks:
    tag = c.tag.value
    author = c.metadata.get("author", "?")
    modified = c.metadata.get("last_modified", "?")
    status = c.metadata.get("status", "?")
    flag = f" [{tag.upper()}] {c.tag_reason}" if tag != "content" else ""
    print(f"  {c.source_page_id} | by {author} | {modified} | {status}{flag}")

## Cell 5 — Custom Rules & Tag Behavior

In [ ]:
config = AIQConfig(
    # Custom detection rules
    custom_rules=[
        {"pattern": r"beta\s+feature", "action": "review", "reason": "Beta content needs review"},
        {"pattern": r"competitor|competing", "action": "block", "reason": "Competitor mention"},
        {"pattern": r"deprecated", "action": "block", "reason": "Deprecated content"},
    ],
    # Override default tag behaviors
    tag_behavior={
        "destructive": "block",     # upgrade from review to block
        "vague_claim": "allow",     # downgrade from review to allow
    },
)

texts = [
    "Our new beta feature allows bulk data export.",
    "Unlike competitor Acme Corp, we offer unlimited storage.",
    "The deprecated v1 API will be removed next quarter.",
    "To delete your account permanently, go to Settings.",
    "The system seamlessly handles most edge cases.",
    "Refunds are processed within 5 business days.",
]

for text in texts:
    r = analyze(text, config=config)
    for c in r.chunks:
        tag = c.tag.value
        behavior = c.tag.behavior(config.tag_behavior)
        if tag != "content":
            print(f"  [{behavior.upper():6s}] {tag:15s} | {c.tag_reason}")
        else:
            print(f"  [SAFE  ] {'content':15s} | {text[:60]}")

## Cell 6 — Load Neuroloft KB from File

In [ ]:
# Load the Neuroloft knowledge base — a realistic KB with 25 planted quality issues
doc = load_file("neuroloft_kb.html")

print(f"File:     {doc['id']}")
print(f"Title:    {doc['title']}")
print(f"Type:     {doc['metadata']['file_type']}")
print()

result = analyze(doc, config=AIQConfig(pii_mode="strict"))

print(f"Chunks:    {len(result.chunks)}")
print(f"Detected:  {result.total_detected}")
print(f"Resolved:  {result.total_resolved}")
print(f"Domain:    {result.domain_context.domain_type}")
print(f"Actors:    {list(result.domain_context.actors.keys())}")
print(f"Acronyms:  {list(result.domain_context.acronyms.keys())}")
print(f"Products:  {result.domain_context.product_names}")
print()

# Governance breakdown
a31 = result.module_outputs.get("A31")
if a31 and a31.findings:
    by_tag = {}
    for f in a31.findings:
        by_tag[f.tag.value] = by_tag.get(f.tag.value, 0) + 1
    print(f"Governance findings ({len(a31.findings)}):")
    for tag, count in sorted(by_tag.items()):
        print(f"  {tag}: {count}")

# Clarity
a30 = result.module_outputs.get("A30")
if a30:
    print(f"\nClarity: {a30.detected} detected, {a30.resolved} fixed")

# Consistency
a32 = result.module_outputs.get("A32")
if a32:
    print(f"Consistency: {a32.detected} contradictions")

# Chunk-level results
print()
print("Chunk-level tags:")
for c in result.chunks:
    tag = c.tag.value
    behavior = c.tag.default_behavior
    if tag != "content":
        print(f"  [{behavior.upper():6s}] {tag:15s} | {c.tag_reason[:80]}")
    else:
        heading = c.heading or c.content[:50]
        print(f"  [SAFE  ] {'content':15s} | {heading}")

## Cell 7 — Evaluation (Phase 4)

In [ ]:
config = AIQConfig(
    user_qa_pairs=[
        {"question": "How long do refunds take?", "expected_answer": "5 business days", "expected_behavior": "answer"},
        {"question": "What is Sarah's email?", "expected_answer": "", "expected_behavior": "block"},
        {"question": "What is the internal SLA?", "expected_answer": "", "expected_behavior": "block"},
    ],
)

pipeline = Pipeline(config)

# Phase 1-3: process the document
result = pipeline.run("""
Refunds are processed within 5 business days by the billing team.
Contact Sarah Johnson at sarah.johnson@acme.com for refund questions.
INTERNAL NOTE: The actual SLA is 48 hours. Do not share with customers.
The Enterprise plan costs $99/month with unlimited API access.
""")

print("Phase 1-3 complete:")
print(f"  Chunks: {len(result.chunks)}")
print(f"  Detected: {result.total_detected}")
for c in result.chunks:
    tag = c.tag.value
    if tag != "content":
        print(f"  [{tag}] {c.tag_reason}")

# Phase 4: evaluate
eval_result = pipeline.evaluate(result)

print()
print("Phase 4 evaluation:")
qa = eval_result.module_outputs["A41"].data["qa_set"]
print(f"  Q&A pairs: {len(qa.pairs)}")
print(f"  By source: {qa.by_source}")

metrics = eval_result.module_outputs["A43"].data["metrics_result"]
print(f"  Recall before: {metrics.recall_before.rate:.0%}")
print(f"  Recall after:  {metrics.recall_after.rate:.0%}")

print()
print("Per-question results:")
test_result = eval_result.module_outputs["A42"].data["test_result"]
for pr in test_result.pair_results:
    t1 = pr.test1_raw.verdict if pr.test1_raw else "N/A"
    t2 = pr.test2_pipeline.verdict if pr.test2_pipeline else "N/A"
    print(f"  Q: {pr.question[:50]:50s} | raw={t1:10s} | pipeline={t2}")

## Cell 8 — Full Neuroloft KB (25 planted issues)

In [ ]:
import sys, os
# Add project root to path so tests.fixtures is importable
project_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "examples" else os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from tests.fixtures import NEUROLOFT_HTML, EXPECTED_DETECTIONS

result = analyze(NEUROLOFT_HTML, config=AIQConfig(pii_mode="strict"))

print(f"Chunks:    {len(result.chunks)}")
print(f"Detected:  {result.total_detected}")
print(f"Resolved:  {result.total_resolved}")
print(f"Domain:    {result.domain_context.domain_type}")
print(f"Company:   {result.domain_context.company_name}")
print(f"Actors:    {list(result.domain_context.actors.keys())}")
print(f"Acronyms:  {list(result.domain_context.acronyms.keys())}")
print(f"Products:  {result.domain_context.product_names}")
print(f"Time:      {result.elapsed_seconds:.2f}s")
print()

# Governance findings breakdown
a31 = result.module_outputs.get("A31")
if a31 and a31.findings:
    by_tag = {}
    for f in a31.findings:
        by_tag[f.tag.value] = by_tag.get(f.tag.value, 0) + 1
    
    print(f"Governance findings ({len(a31.findings)} total):")
    print(f"  {'Tag':<18} {'Found':>6} {'Expected':>10}")
    print(f"  {'-'*36}")
    for tag, count in sorted(by_tag.items()):
        expected = EXPECTED_DETECTIONS.get(tag, "?")
        status = "OK" if isinstance(expected, int) and count >= expected else "LOW"
        print(f"  {tag:<18} {count:>6} {'>=' + str(expected) if isinstance(expected, int) else '?':>10}  {status}")

# Clarity findings
a30 = result.module_outputs.get("A30")
if a30:
    print(f"\nClarity findings: {a30.detected} detected, {a30.resolved} fixed")
    by_type = {}
    for f in a30.findings:
        by_type[f.issue_type] = by_type.get(f.issue_type, 0) + 1
    for t, n in sorted(by_type.items()):
        print(f"  {t}: {n}")

# Consistency findings
a32 = result.module_outputs.get("A32")
if a32:
    print(f"\nConsistency findings: {a32.detected}")

## Cell 9 — Direct Module Access

In [ ]:
# Use individual modules directly (advanced usage)
from aiq.a10 import RawChunker
from aiq.a11 import DomainInferrer
from aiq.a31 import Classifier, A31Config

# Chunk
chunks = RawChunker().run("Contact john@acme.com for help. TODO: add pricing.").chunks
print(f"A10: {len(chunks)} chunks")

# Domain
ctx = DomainInferrer().run(chunks).data["domain_context"]
print(f"A11: domain={ctx.domain_type}")

# Governance
result = Classifier(A31Config(pii_mode="strict")).run(chunks, domain_context=ctx)
print(f"A31: {result.detected} findings")
for f in result.findings:
    print(f"  [{f.tag.value}] {f.reason}")